# JAX-ALFA Simulation Run Log

Scans every `run.log` file in the three campaign result trees listed below (sandbox and older trees are excluded) and reports when each simulation
was run, how long it took, and on which platform. Only finished runs whose results
have been downloaded to this repository are listed; the newest runs are at the top.

**Platform** is the results tree: `examples` (UAlbany DGX, A100), `examples_A6000ada` (HAL, RTX 6000 Ada), `examples_H100` (Empire AI, H100).

In [1]:
from IPython.display import display, Markdown
from datetime import datetime, timezone
display(Markdown(f"*Last run: {datetime.now(timezone.utc).strftime('%B %d, %Y at %H:%M UTC')}*"))

*Last run: November 20, 2026 at 00:09 UTC*

In [2]:
import os, re
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone, timedelta
from IPython.display import display, HTML

def find_repo_root():
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'examples').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate jaxalfa repository root')

def read_log_info(log_path, tail_bytes=16384):
    """Read elapsed time and CFL values from the tail of a run.log."""
    info = {'elapsed': None, 'cfl_last': None, 'cfl_max': None}
    try:
        size = os.path.getsize(log_path)
        with open(log_path, 'rb') as f:
            f.seek(max(0, size - tail_bytes))
            tail = f.read().decode('utf-8', errors='ignore')
        m = re.search(r'Total Elapsed Time:\s+([\d.]+)\s+seconds', tail)
        if m:
            info['elapsed'] = float(m.group(1))
        for m in re.finditer(r'Current CFL:\s+([\d.]+)', tail):
            info['cfl_last'] = float(m.group(1))
        for m in re.finditer(r'CFLmax:\s+([\d.]+)', tail):
            info['cfl_max'] = float(m.group(1))
    except Exception:
        pass
    return info

repo = find_repo_root()
print(f'Repository root: {repo}')

Repository root: /Users/sukantabasu/Dropbox/Codes/LES/JAX-ALFA/jaxalfa


In [3]:
# Campaign result trees only (sandbox and older platform trees are excluded)
PLATFORMS = ['examples', 'examples_A6000ada', 'examples_H100']

rows = []

for log_path in sorted(p for t in PLATFORMS for p in repo.glob(f'{t}/*/runs/*/run.log')):
    rel      = log_path.relative_to(repo).parts
    platform = rel[0]
    case     = rel[1]
    run      = rel[3]

    mtime     = os.path.getmtime(log_path)
    completed = datetime.fromtimestamp(mtime, tz=timezone.utc)
    info      = read_log_info(log_path)
    elapsed   = info['elapsed']

    if elapsed is not None:
        started   = completed - timedelta(seconds=elapsed)
        wall_time = str(timedelta(seconds=int(elapsed)))
    else:
        started   = None
        wall_time = '—'

    rows.append({
        'Platform'       : platform,
        'Case'           : case,
        'Run'            : run,
        'Started (UTC)'  : started.strftime('%Y-%m-%d %H:%M') if started else '—',
        'Completed (UTC)': completed.strftime('%Y-%m-%d %H:%M'),
        'Wall Time'      : wall_time,
        'CFL (last)'     : f"{info['cfl_last']:.4f}" if info['cfl_last'] is not None else '—',
        'CFL (max)'      : f"{info['cfl_max']:.4f}"  if info['cfl_max']  is not None else '—',
        '_completed_dt'  : completed,
    })

df_all = pd.DataFrame(rows)
print(f'{len(df_all)} runs found across {df_all["Platform"].nunique()} platform(s)')

89 runs found across 2 platform(s)


## By Completion Date (newest first)

The most recently completed runs appear at the top.

In [4]:
display_cols = ['Platform', 'Case', 'Run', 'Started (UTC)', 'Completed (UTC)', 'Wall Time', 'CFL (last)', 'CFL (max)']

df_by_date = (
    df_all.sort_values('_completed_dt', ascending=False)
          .reset_index(drop=True)
          [display_cols]
)
df_by_date.index += 1
display(df_by_date.style.set_properties(**{'text-align': 'left'}))

,Platform,Case,Run,Started (UTC),Completed (UTC),Wall Time,CFL (last),CFL (max)
1,examples,SBL_GABLS4,128x128x128_LASDD_WL_DP,2026-11-19 18:38,2026-11-19 23:54,5:15:31,0.0700,0.0730
2,examples,SBL_GABLS1,128x128x128_LAD_SM_SP,2026-11-19 21:08,2026-11-19 23:43,2:34:59,0.0790,0.0870
3,examples,NBL_A94,128x128x128_LAD_SM_DP,2026-11-19 20:58,2026-11-19 22:47,1:49:03,0.0880,0.0960
4,examples,SBL_GABLS3,128x128x128_LASDD_WL_DP,2026-11-19 17:53,2026-11-19 22:37,4:43:20,0.0650,0.0930
5,examples_A6000ada,SBL_GABLS4,64x64x64_LASDD_SM_SP,2026-11-19 20:59,2026-11-19 22:35,1:36:17,0.0340,0.0360
6,examples,NBL_A94,128x128x128_LAD_SM_SP,2026-11-19 20:53,2026-11-19 22:06,1:13:12,0.0850,0.0980
7,examples,SBL_GABLS4,128x128x128_LASDD_WL_SP,2026-11-19 18:18,2026-11-19 22:04,3:45:57,0.0700,0.0740
8,examples,DC_Wangara,128x128x128_LASDD_WL_DP,2026-11-19 19:28,2026-11-19 21:36,2:08:22,0.0550,0.1000
9,examples,SBL_GABLS1,128x128x128_LASDD_WL_DP,2026-11-19 17:33,2026-11-19 21:20,3:46:59,0.0780,0.0980
10,examples,SBL_GABLS3,128x128x128_LASDD_WL_SP,2026-11-19 17:48,2026-11-19 21:07,3:18:45,0.0650,0.0930


## By Platform → Case → Run

Same data grouped alphabetically by platform, case, and run name.

In [5]:
df_by_name = (
    df_all.sort_values(['Platform', 'Case', 'Run'])
          .reset_index(drop=True)
          [display_cols]
)
df_by_name.index += 1
display(df_by_name.style.set_properties(**{'text-align': 'left'}))

,Platform,Case,Run,Started (UTC),Completed (UTC),Wall Time,CFL (last),CFL (max)
1,examples,CBL_N91,128x128x128_LAD_SM_DP,2026-11-19 20:33,2026-11-19 20:51,0:18:03,0.0260,0.0330
2,examples,CBL_N91,128x128x128_LAD_SM_SP,2026-11-19 20:18,2026-11-19 20:30,0:12:00,0.0310,0.0330
3,examples,CBL_N91,128x128x128_LASDD_SM_DP,2026-11-19 12:42,2026-11-19 13:05,0:22:51,0.0280,0.0340
4,examples,CBL_N91,128x128x128_LASDD_SM_SP,2026-11-19 12:42,2026-11-19 12:57,0:14:46,0.0270,0.0350
5,examples,CBL_N91,128x128x128_LASDD_WL_DP,2026-11-19 16:07,2026-11-19 16:25,0:17:49,0.0330,0.0330
6,examples,CBL_N91,128x128x128_LASDD_WL_SP,2026-11-19 16:02,2026-11-19 16:14,0:11:54,0.0280,0.0330
7,examples,CBL_N91,64x64x64_LAD_SM_DP,2026-11-19 09:36,2026-11-19 09:44,0:07:49,0.0130,0.0140
8,examples,CBL_N91,64x64x64_LAD_SM_SP,2026-11-19 09:26,2026-11-19 09:34,0:07:52,0.0120,0.0140
9,examples,CBL_N91,64x64x64_LAD_WL_DP,2026-11-19 11:07,2026-11-19 11:15,0:08:09,0.0120,0.0140
10,examples,CBL_N91,64x64x64_LAD_WL_SP,2026-11-19 11:07,2026-11-19 11:15,0:08:03,0.0120,0.0140


## Summary by Platform

In [6]:
summary = (
    df_all.groupby('Platform')
          .agg(
              Runs        = ('Run', 'count'),
              Oldest_Run  = ('_completed_dt', lambda x: x.min().strftime('%Y-%m-%d')),
              Newest_Run  = ('_completed_dt', lambda x: x.max().strftime('%Y-%m-%d')),
          )
          .rename(columns={'Oldest_Run': 'Oldest Run', 'Newest_Run': 'Newest Run'})
)
display(summary)

,Runs,Oldest Run,Newest Run
Platform,,,
examples,77,2026-11-19,2026-11-19
examples_A6000ada,12,2026-11-19,2026-11-19
